# Pearls AQI Predictor — Exploratory Data Analysis

This notebook analyzes historical AQI and weather data for Karachi to:
- Identify seasonal and diurnal patterns
- Understand correlations between features
- Determine optimal lag windows (ACF/PACF)
- Detect outliers and data quality issues
- Compute persistence baseline RMSE

**Prerequisites:** MongoDB connection details must be set in `.env` before running this notebook (the feature store lives in MongoDB Atlas — Hopsworks/Vertex AI are *not* used).

```bash
cp .env.example .env  # then fill in MONGODB_URI and MONGODB_DB_NAME
```

The data is loaded directly from the `aqi_features` collection via `fetch_training_data()`, so no AQICN / OpenWeather keys are needed just to run this analysis.

Run from the `notebooks/` directory, or adjust `sys.path` in the first code cell accordingly.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

# Load MongoDB credentials from the project-root .env so fetch_training_data() works
from dotenv import load_dotenv
load_dotenv(Path('..') / '.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error

sns.set_style('darkgrid')
pd.set_option('display.max_columns', 50)

# Export figures for the LaTeX report (../report/figures). Requires `kaleido`
# for Plotly PNG export:  pip install kaleido
FIG_DIR = Path('..') / 'report' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_plotly(fig, name, width=1000, height=600):
    """Write a Plotly figure to report/figures/<name>.png (needs kaleido)."""
    fig.write_image(str(FIG_DIR / f'{name}.png'), width=width, height=height, scale=2)

print('Libraries loaded — figures will be exported to', FIG_DIR.resolve())

## 1. Load Data from MongoDB Feature Store

In [ ]:
from src.feature_pipeline.store_features import fetch_training_data

df = fetch_training_data()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# day_of_week is not stored as a column (only one-hot dow_0..dow_6 are stored);
# reconstruct it from timestamp for the EDA visualisations below.
df['day_of_week'] = df['timestamp'].dt.dayofweek

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Date range: {df["timestamp"].min()} -> {df["timestamp"].max()}')
df.head()

### Feature / target alignment with the models

These are the **exact rows the training pipeline feeds its models** — `fetch_training_data()` reads the same `aqi_features` collection used in training. The models are **multi-output (MIMO)**: every model predicts the next three days *simultaneously* from one feature vector, so there are three target columns:

| Target | Meaning |
|--------|---------|
| `aqi_24h` | AQI 24 h ahead (Day 1) |
| `aqi_48h` | AQI 48 h ahead (Day 2) |
| `aqi_72h` | AQI 72 h ahead (Day 3) |

The predictors span ~50 engineered features grouped into: raw pollutants, meteorology, time (basic + Fourier), lags (1 h → 168 h / 7 days), rolling statistics, and physics/cross features. The cell below confirms which of those groups are present in the loaded frame so the EDA stays in sync with the feature pipeline.

In [ ]:
from src.config import FEATURE_GROUPS, FORECAST_HOURS

# One representative column per engineered feature group — confirms the loaded
# frame carries the same groups the training pipeline enables.
group_probe = {
    'time_basic':      'hour',
    'time_fourier':    'hour_sin',
    'lag_features':    'aqi_lag_24h',
    'rolling_stats':   'rolling_mean_24h',
    'physics_derived': 'mixing_height_proxy',
    'cross_feature':   'aqi_change_rate',
    'raw_pollutants':  'pm25',
    'meteorology':     'temperature',
}
print('Feature-group coverage in the loaded frame:')
for g, col in group_probe.items():
    print(f'  {g:16s} enabled={str(FEATURE_GROUPS.get(g)):5s} '
          f'probe={col:20s} present={col in df.columns}')

targets = [f'aqi_{h}h' for h in FORECAST_HOURS]
print(f'\nForecast targets {targets} present:',
      [t for t in targets if t in df.columns])
print(f'Total columns: {df.shape[1]}')

## 2. Basic Statistics & Data Quality

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum().sort_values(ascending=False).head(20))
print()
print('=== Descriptive Stats ===')
df[['aqi', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'temperature', 'humidity', 'wind_speed']].describe()

## 3. AQI Time Series

In [ ]:
fig = px.line(df, x='timestamp', y='aqi',
              title='AQI Over Time — Karachi',
              labels={'aqi': 'AQI', 'timestamp': 'Date'},
              template='plotly_dark')
# Add AQI zone bands
for lo, hi, label, color in [(0,50,'Good','rgba(0,228,0,0.08)'),
                               (51,100,'Moderate','rgba(255,255,0,0.06)'),
                               (101,150,'Unhealthy (Sensitive)','rgba(255,126,0,0.08)'),
                               (151,200,'Unhealthy','rgba(255,0,0,0.08)'),
                               (201,500,'Hazardous','rgba(126,0,35,0.1)')]:
    fig.add_hrect(y0=lo, y1=hi, fillcolor=color, line_width=0, annotation_text=label, annotation_position='right')
save_plotly(fig, 'aqi_timeseries', height=500)
fig.show()

## 4. Diurnal & Seasonal Patterns

In [ ]:
# AQI by hour of day
if 'hour' in df.columns:
    hourly = df.groupby('hour')['aqi'].agg(['mean', 'std']).reset_index()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hourly['hour'], y=hourly['mean'] + hourly['std'],
                             fill=None, mode='lines', line_color='rgba(74,144,217,0)', showlegend=False))
    fig.add_trace(go.Scatter(x=hourly['hour'], y=hourly['mean'] - hourly['std'],
                             fill='tonexty', mode='lines', line_color='rgba(74,144,217,0)',
                             fillcolor='rgba(74,144,217,0.2)', name='±1 Std'))
    fig.add_trace(go.Scatter(x=hourly['hour'], y=hourly['mean'], mode='lines+markers',
                             name='Mean AQI', line=dict(color='#4a90d9', width=2)))
    fig.update_layout(title='Mean AQI by Hour of Day', template='plotly_dark',
                      xaxis_title='Hour', yaxis_title='AQI')
    save_plotly(fig, 'aqi_by_hour', height=450)
    fig.show()

In [ ]:
# AQI heatmap: hour × day_of_week
if 'hour' in df.columns and 'day_of_week' in df.columns:
    pivot = df.pivot_table(values='aqi', index='hour', columns='day_of_week', aggfunc='mean')
    pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    fig = px.imshow(pivot, title='Mean AQI: Hour × Day of Week',
                    color_continuous_scale='RdYlGn_r', template='plotly_dark',
                    labels={'color': 'AQI'})
    save_plotly(fig, 'aqi_hour_dow_heatmap', width=700, height=600)
    fig.show()

## 5. Correlation Heatmap

In [ ]:
# Correlation across one representative feature per model feature-group, plus the
# three forecast targets — so the heatmap reflects what the models actually consume
# and reveals which predictors track the 24h/48h/72h targets.
key_cols = [
    'aqi',                                              # current AQI
    'pm25', 'pm10', 'o3', 'no2', 'so2', 'co',           # raw pollutants
    'temperature', 'humidity', 'pressure', 'wind_speed',# meteorology
    'aqi_lag_1h', 'aqi_lag_24h', 'aqi_lag_168h',        # lags (short/daily/weekly)
    'rolling_mean_24h', 'rolling_std_24h',              # rolling stats
    'aqi_change_rate', 'pressure_anomaly',              # cross features
    'aqi_24h', 'aqi_48h', 'aqi_72h',                    # MIMO forecast targets
]
key_cols = [c for c in key_cols if c in df.columns]

corr = df[key_cols].corr()
fig = px.imshow(corr, title='Feature ↔ Target Correlation Matrix',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                template='plotly_dark', text_auto='.2f', aspect='auto')
fig.update_layout(width=900, height=820)
save_plotly(fig, 'correlation_matrix', width=900, height=820)
fig.show()

## 5b. Forecast-Target Distributions

The three targets `aqi_24h / aqi_48h / aqi_72h` are future values of the same AQI series, so they should share the current AQI's shape (right-skewed with a long hazardous tail). Confirming this supports training a **single multi-output (MIMO) model** across all three horizons rather than three separate models.

In [ ]:
target_cols = ['aqi'] + [c for c in ['aqi_24h', 'aqi_48h', 'aqi_72h'] if c in df.columns]
colors = {'aqi': '#4a90d9', 'aqi_24h': '#a6e3a1', 'aqi_48h': '#f5c842', 'aqi_72h': '#f38ba8'}
fig = go.Figure()
for col in target_cols:
    fig.add_trace(go.Histogram(x=df[col].dropna(), name=col, nbinsx=60,
                               opacity=0.55, marker_color=colors.get(col)))
fig.update_layout(barmode='overlay', title='Current AQI vs Forecast Targets (24h / 48h / 72h)',
                  template='plotly_dark', xaxis_title='AQI', yaxis_title='Count')
save_plotly(fig, 'target_distributions', height=500)
fig.show()
print(df[target_cols].describe().round(1))

## 6. ACF / PACF — Optimal Lag Selection

In [ ]:
aqi_clean = df['aqi'].dropna()
# Show up to 168h (7 days) so the weekly cycle behind aqi_lag_168h is visible.
max_lags = min(168, len(aqi_clean) // 2 - 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(aqi_clean, lags=max_lags, ax=ax1, title=f'ACF — AQI ({max_lags}h lags)')
plot_pacf(aqi_clean, lags=max_lags, ax=ax2, title=f'PACF — AQI ({max_lags}h lags)', method='ywmle')
# Mark the daily (24h) and weekly (168h) lags the feature pipeline encodes
for ax in (ax1, ax2):
    for L in (24, 48, 72, 168):
        if L <= max_lags:
            ax.axvline(L, color='#f5c842', ls='--', lw=0.8, alpha=0.6)
plt.tight_layout()
plt.savefig(FIG_DIR / 'acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashed lines mark 24/48/72/168h — the lag horizons used as model features.')
print('Strong spikes at 24h and 168h justify the daily and weekly lag features.')

## 7. Outlier Detection (IQR)

In [ ]:
for col in ['aqi', 'pm25', 'pm10', 'temperature']:
    if col not in df.columns:
        continue
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 3*IQR, Q3 + 3*IQR
    n_outliers = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col:20s}: {n_outliers} outliers ({n_outliers/len(df)*100:.1f}%) | range=[{lo:.1f}, {hi:.1f}]')

## 8. Persistence Baseline RMSE

In [ ]:
# Persistence baseline for each forecast horizon the models predict (MIMO).
# Persistence = "tomorrow equals today": predict aqi_{h}h with the current aqi.
print('Persistence baseline (predict current AQI for every horizon):\n')
print(f'{"Horizon":>8} | {"RMSE":>8} | {"MAE":>8} | {"n":>7}')
print('-' * 40)
for h in [24, 48, 72]:
    col = f'aqi_{h}h'
    if col not in df.columns:
        continue
    valid = df[['aqi', col]].dropna()
    if valid.empty:
        continue
    persistence_preds = valid['aqi'].values
    actual = valid[col].values
    rmse = np.sqrt(mean_squared_error(actual, persistence_preds))
    mae = np.mean(np.abs(actual - persistence_preds))
    print(f'{h:>6}h | {rmse:>8.2f} | {mae:>8.2f} | {len(valid):>7}')

print('\nThese are the scores the trained models must beat — a positive Skill Score')
print('means the model improves on persistence for that horizon. RMSE rises with the')
print('horizon, which is why the 48h/72h targets are harder than 24h.')

## 9. Pollutant Distributions

In [ ]:
pollutants = [c for c in ['pm25', 'pm10', 'o3', 'no2', 'so2', 'co'] if c in df.columns]
fig = make_subplots(rows=2, cols=3, subplot_titles=pollutants)
for i, p in enumerate(pollutants):
    r, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Histogram(x=df[p].dropna(), name=p, nbinsx=50,
                               marker_color='#4a90d9', opacity=0.8), row=r, col=c)
fig.update_layout(title='Pollutant Distributions (µg/m³)',
                  template='plotly_dark', showlegend=False, height=500)
save_plotly(fig, 'pollutant_distributions', height=500)
fig.show()
print('Note: PM2.5 and PM10 are right-skewed → log(x+1) transform is applied before training')